In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import html
import ftfy
import string
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
n_placeholders_source = df['source'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
print(f"Number of placeholders (\\N): {n_placeholders_source}")
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 294
Relevant Sources:
Index(['Yahoo', 'Reuters', 'BBC', 'New', 'Washington', 'RedNova', 'Boston',
       'CNN', 'CNET', 'Topix.Net', 'Guardian', 'Motley', 'Register',
       'International', 'Forbes', 'Time', 'ABC', 'InfoWorld', 'San', 'Wired',
       'Xinhua', 'Computerworld', 'News', 'CSMonitor', 'PCWorld', 'Bloomberg',
       'Seattle', 'Ananova', '\N', 'Syfy.com', 'Voice', 'USA', 'Independent',
       'Scotsman', 'CBS', 'Rediff', 'Times', 'Channel', 'CBC', 'Newsday',
       'Newsweek', 'Houston', 'Australian', 'Daily', 'Telegraph.co.uk', 'ESPN',
       'Canada.com', 'BCC', 'Sports', 'Search', 'Chicago', 'Turkish', 'CNN/SI',
       'MSNBC', 'London', 'National', 'Financial', 'Toronto', 'Indianapolis',
       'Melbourne', 'Christian', 'Detroit', 'ZDNet.com', 'CTV', 'PC', 'ic',
       'NEWS.com.au', 'RTE', 'Scotland', 'Hindustan', 'NPR', 'Al-Jazeera',
       'Information', 'IPS', 'TechNewsWorld', 'News24', 'spo

### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(20))

Number of NaN rows: 1
Number of empty rows: 2
Number of placeholders (\N): 0
Titles Sample:
Id
2220                                         No. 22 Va. Tech Rallies Past Ga. Tech (AP)
51596                              Senegal's religious leader dead at 92 \\n    (AP)\\n
14073                                          Abbas weighs early poll after talks fail
63189                                               Iowa Winners Woo Opponents' Donors 
24858                            Food poisoning can be long-term problem \\n    (AP)\\n
20218                                                 Serb general faces Hague tribunal
71719                                                         Palm warns of profit drop
12848                                     FBI Probing Suspected Israeli Spy at Pentagon
19675    Goodell wants the world passionate about American football \\n    (Reuters)\\n
57046                                                London Times goes strictly tabloid
41632                    

### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of empty rows: 7
Number of placeholders (\N): 1874
Articles Sample
Id
23834    Roger Clemens and his ex-trainer stick to different stories, amid withering questioning on Hill.<br style="clear: both;"/>\\n  <img alt="" style="border: 0; height:1px; width:1px;" border="0" src="http://www.pheedo.com/img.phdo?i=92e723ca4e6a80cb4c079cff7d457fe3" height="1" width="1"/>\\n<img src="http://www.pheedo.com/feeds/tracker.php?i=92e723ca4e6a80cb4c079cff7d457fe3" style="display: none;" border="0" height="1" width="1" alt=""/>\\n<p><a href="http://rss.csmonitor.com/~a/feeds/arts?a=HWHZiA"><img src="http://rss.csmonitor.com/~a/feeds/arts?i=HWHZiA" border="0"></img></a></p><div class="feedflare">\\n<a href="http://rss.csmonitor.com/~f/feeds/arts?a=yWk818E"><img src="http://rss.csmonitor.com/~f/feeds/arts?i=yWk818E" border="0"></img></a> <a href="http://rss.csmonitor.com/~f/feeds/arts?a=mOFeNqE"><img src="http://rss.csmonitor.com/~f/feeds/arts?i=mOFeNqE" border="0"></img></a

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number of articles with PageRank 5: 73891
Id
77209    5
43680    5
59208    5
7534     5
31615    5
48       5
57204    5
20355    5
10517    5
38599    4
Name: page_rank, dtype: int64


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
52560    0000-00-00 00:00:00
72296    2007-03-07 23:38:56
19475    0000-00-00 00:00:00
70078    2007-09-16 22:51:33
58186    0000-00-00 00:00:00
61877    0000-00-00 00:00:00
12940    0000-00-00 00:00:00
43695    2006-12-13 02:32:51
25512    2008-01-22 17:19:48
18309    2008-01-29 17:41:48
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    df['has_date'] = df['dt_obj'].notna().astype(int)
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)
    df['hour'] = df['dt_obj'].dt.hour.fillna(-1).astype(int)

    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)
new_cols = ['has_date', 'year', 'month', 'day_of_week', 'hour']
print(f"Nuove colonne aggiunte: {new_cols}\n")
print("Test new timestamp features:\n")
print(df[new_cols].sample(10))

Nuove colonne aggiunte: ['has_date', 'year', 'month', 'day_of_week', 'hour']

Test new timestamp features:

       has_date  year  month  day_of_week  hour
Id                                             
52483         1  2007      2            5     6
15278         1  2007      7            1    14
39854         0    -1     -1           -1    -1
18262         1  2008      1            3     6
67556         0    -1     -1           -1    -1
45951         1  2007      6            4     3
5882          1  2007     12            3    20
49603         1  2006     11            5     3
2404          1  2004      8            3    19
15284         0    -1     -1           -1    -1


### *Title* feature stemming

In [9]:
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english'))

def clean_title(text):
    if pd.isna(text) or text == "": return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = text.lower() 
    
    text = re.sub(r'(?:\\n|\s)*\(.*?\)\W*$', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    # tag Money
    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s*(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    # tag Percentage
    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    # tag Score
    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    #tag Date
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z_]', ' ', text)

    words = text.split()
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(meaningful_words)

def test_simple(titles_series, cleaner_func, n):
    sample = titles_series.sample(n)
    print(f"Test on {n} random samples\n")
    
    for idx, text in sample.items():
        cleaned = cleaner_func(text)
        print(f"Original text:  {text}")
        print(f"Processed text: {cleaned}")
        print("-" * 50)

test_simple(df['title'], clean_title, n=10)

Test on 10 random samples

Original text:  Intel Intros New Flash Software (NewsFactor)
Processed text: intel intro new flash softwar
--------------------------------------------------
Original text:  Baghdatis tops Ljubicic to win Zagreb \
    (AP)\

Processed text: baghdati top ljubic win zagreb
--------------------------------------------------
Original text:  New video-game league seeks mass appeal
Processed text: new video game leagu seek mass appeal
--------------------------------------------------
Original text:  Illinois Helps Residents Import Prescription Drugs
Processed text: illinoi help resid import prescript drug
--------------------------------------------------
Original text:  From Kiev to M Street
Processed text: kiev street
--------------------------------------------------
Original text:  He's living for the moment
Processed text: live moment
--------------------------------------------------
Original text:  China defends its role in Africa ahead of G8
Processed text

### *Article* feature stemming

In [10]:
def clean_article(text):
    if pd.isna(text) or text == "" or str(text).strip() == "\\N": 
        return ""
    text = str(text)

    text = re.sub(r'<[^>]+>', ' ', text)
    text = ftfy.fix_text(text)
    text = text.strip()
    
    text = re.sub(r'^\s*[A-Z][\w\s,\.\(\)]{0,50}\s*--\s*', '', text)
    
    agencies_pattern = r'(?i)^\s*.*?\b(reuters|afp|ap|upi|bloomberg|bbc|cnn|blog)\b.*?\s*[-:–—]\s*'
    text = re.sub(agencies_pattern, '', text)
    text = re.sub(r'^\s*[A-Z][^\.\?!]{2,50}\s+[-–—]\s+', '', text)
    text = re.sub(r'(?i)^by\s+[a-z\s\.,]+\s{2,}', '', text)

    text = text.lower()
    text = text[:500]

    # tag Money
    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s*(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    # tag Percentage
    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    # tag Score
    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    #tag Date
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z_]', ' ', text)

    words = text.split()
    meaningful_words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(meaningful_words)

def test_simple_article(article_series, cleaner_func, n):
    sample = article_series.sample(n)
    print(f"Test on {n} random samples\n")
    for idx, text in sample.items():
        print(f"Original text (First 200 char): {str(text)}") 
        print(f"Processed text: {cleaner_func(text)}")
        print("-" * 50)

test_simple_article(df['article'], clean_article, n=10)

Test on 10 random samples

Original text (First 200 char): AP - A 90-year-old nursing home patient died from the stomach flu last year, marking the first time U.S. health officials confirmed that the highly contagious bug is sometimes fatal.
Processed text: year old nurs home patient die stomach flu last year mark first time health offici confirm high contagi bug sometim fatal
--------------------------------------------------
Original text (First 200 char): Like other 18-year-olds, Mary Ann Foco was listening to Elton John and Al Green and putting her hopes in a college degree. But her seemingly healthy body was quietly failing her.
Processed text: like year old mari ann foco listen elton john green put hope colleg degre seem healthi bodi quiet fail
--------------------------------------------------
Original text (First 200 char): The Voyager probes are now the farthest manmade objects from Earth, continuing to relay data 30 years after their launch. We've assembled some of the more a

### *Title + Article* features merge

In [11]:
print(f"Starting shape: {df.shape}")
print(f"Starting columns: {df.columns.tolist()}")
df['title_clean'] = df['title'].apply(clean_title)
df['article_clean'] = df['article'].apply(clean_article)
df['text_combined'] = (df['title_clean'].fillna('') + " " + df['article_clean'].fillna('')).str.strip()

n_empty = (df['text_combined'] == "").sum()
print(f"Removing {n_empty} rows with empty text")
df = df[df['text_combined'] != ""]

cols_to_drop = ['title', 'article', 'title_clean', 'article_clean']
df = df.drop(columns=cols_to_drop)

print(f"Final shape: {df.shape}")
print(f"Actual columns: {df.columns.tolist()}")
print("Example of title + article combined:")
print(df['text_combined'].iloc[0])

Starting shape: (79997, 10)
Starting columns: ['source', 'title', 'article', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'hour']
Removing 3 rows with empty text
Final shape: (79994, 9)
Actual columns: ['source', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'hour', 'text_combined']
Example of title + article combined:
opec boost nigeria oil revenu tag_money bpd organis petroleum export countri opec hike offici output one million barrel per day effect novemb nigeria get barrel per day per cent new quota
